# 🦕 DINO SDK v2.3.1 - Teste Correção Base Parameters

Este notebook testa especificamente a correção dos base_parameters no DINO SDK v2.3.1.

**Problema Identificado v2.3.0:**
- Base parameters não apareciam na configuração do job
- Código estava usando classes `Task` e `NotebookTask` em vez de dicionários

**Correção Aplicada v2.3.1:**
- ✅ Mudou para usar `_build_job_settings()` com dicionários
- ✅ Base parameters agora incluídos via dicionário `notebook_task`
- ✅ Resultado inclui `job_config` para debug

**Teste:** Verificar se os base_parameters aparecem corretamente no job criado.

## 🔄 1. Reinstalar SDK v2.3.1

In [ ]:
# Desinstalar versão anterior e instalar v2.3.1
%pip uninstall dino-sdk -y
%pip install /FileStore/wheels/dino_sdk-2.3.1-py3-none-any.whl --force-reinstall
%restart_python

## 📦 2. Importar e Verificar Versão

In [ ]:
# Verificar instalação
from dino_sdk import create_dino_job
import json
from datetime import datetime

print("🦕 DINO SDK v2.3.1 - Teste Base Parameters Fix")
print("=" * 50)
print(f"⏰ {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Verificar se importou corretamente
try:
    # Test import
    from dino_sdk.workflow_manager import WorkflowManager
    from dino_sdk.schema_manager import SchemaManager
    print("✅ Todos os módulos importados com sucesso!")
except ImportError as e:
    print(f"❌ Erro na importação: {e}")

## 🎯 3. Teste Criação de Job com Base Parameters

In [ ]:
print("🎯 TESTE: Criação de Job com Base Parameters")
print("=" * 45)

# Parâmetros de teste
CATALOG_NAME = "data_master_dev_dbw"
SCHEMA_NAME = "bronze"
TABLE_NAME = "test_base_params_v231"

print(f"📋 Configuração do Teste:")
print(f"   📚 Catálogo: {CATALOG_NAME}")
print(f"   🗂️  Schema: {SCHEMA_NAME}")
print(f"   📄 Tabela: {TABLE_NAME}")

# Esperado nos base_parameters
expected_params = {
    "source_path": f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/raw",
    "table_name": TABLE_NAME,
    "catalog_name": CATALOG_NAME,
    "schema_name": SCHEMA_NAME,
    "type_run": "batch"
}

print(f"\n🎯 Base Parameters Esperados:")
for key, value in expected_params.items():
    print(f"   📌 {key}: {value}")

In [ ]:
# Executar criação do job
print(f"\n🚀 Criando job com DINO SDK v2.3.1...")

try:
    # Criar job automatizado (com file arrival trigger)
    result = create_dino_job(
        catalog_name=CATALOG_NAME,
        schema_name=SCHEMA_NAME,
        table_name=TABLE_NAME,
        is_automated=True  # File arrival trigger
    )
    
    print(f"✅ Job criado com sucesso!")
    print(f"🆔 Job ID: {result.get('job_id', 'N/A')}")
    print(f"🏷️  Nome: {result.get('job_name', 'N/A')}")
    print(f"📊 Success: {result.get('success', False)}")
    
    # Verificar se job_config está disponível para debug
    if 'job_config' in result:
        print(f"📋 Job Config: Disponível para análise")
        job_config = result['job_config']
        
        # Verificar estrutura das tasks
        if 'tasks' in job_config and len(job_config['tasks']) > 0:
            print(f"📝 Tasks encontradas: {len(job_config['tasks'])}")
            
            # Analisar primeira task
            task = job_config['tasks'][0]
            print(f"🔑 Task Key: {task.get('task_key', 'N/A')}")
            
            # VERIFICAR BASE PARAMETERS
            if 'notebook_task' in task:
                notebook_task = task['notebook_task']
                print(f"📒 Notebook Path: {notebook_task.get('notebook_path', 'N/A')}")
                print(f"📁 Source: {notebook_task.get('source', 'N/A')}")
                
                # MOMENTO DA VERDADE: Base Parameters
                if 'base_parameters' in notebook_task:
                    params = notebook_task['base_parameters']
                    print(f"\n🎉 BASE PARAMETERS ENCONTRADOS! 🎉")
                    print(f"{'=' * 40}")
                    
                    # Verificar cada parâmetro esperado
                    all_correct = True
                    for key, expected_value in expected_params.items():
                        actual_value = params.get(key, "❌ AUSENTE")
                        
                        if str(actual_value) == str(expected_value):
                            print(f"✅ {key}: {actual_value}")
                        else:
                            print(f"❌ {key}: {actual_value}")
                            print(f"   💡 Esperado: {expected_value}")
                            all_correct = False
                    
                    # Mostrar parâmetros adicionais
                    additional = [k for k in params.keys() if k not in expected_params]
                    if additional:
                        print(f"\n📋 Parâmetros Adicionais:")
                        for key in additional:
                            print(f"   ➕ {key}: {params[key]}")
                    
                    if all_correct:
                        print(f"\n🏆 SUCESSO COMPLETO!")
                        print(f"✅ Todos os base_parameters estão corretos!")
                        print(f"🎯 DINO SDK v2.3.1 funcionando perfeitamente!")
                    else:
                        print(f"\n⚠️  Alguns parâmetros precisam de ajuste")
                        
                else:
                    print(f"\n❌ BASE_PARAMETERS NÃO ENCONTRADOS!")
                    print(f"🔍 Notebook task keys disponíveis:")
                    for key in notebook_task.keys():
                        print(f"   📋 {key}")
            else:
                print(f"❌ notebook_task não encontrado na task!")
                print(f"🔍 Task keys disponíveis:")
                for key in task.keys():
                    print(f"   📋 {key}")
        else:
            print(f"❌ Tasks não encontradas no job_config!")
    else:
        print(f"❌ job_config não disponível no resultado!")
        print(f"🔍 Keys disponíveis no resultado:")
        for key in result.keys():
            print(f"   📋 {key}")
            
except Exception as e:
    print(f"❌ ERRO na criação do job: {str(e)}")
    import traceback
    print(f"🔍 Stack trace:")
    print(traceback.format_exc())

## 📊 4. Análise Detalhada da Estrutura

In [ ]:
# Análise detalhada da estrutura do job
print("📊 ANÁLISE DETALHADA DA ESTRUTURA")
print("=" * 38)

if 'result' in locals() and result.get('success') and 'job_config' in result:
    job_config = result['job_config']
    
    print("🔍 Estrutura Completa do Job Config:")
    
    # Função para imprimir estrutura aninhada
    def print_structure(obj, indent=0, max_depth=3):
        prefix = "  " * indent
        if indent > max_depth:
            print(f"{prefix}[MAX DEPTH REACHED]")
            return
            
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key == 'base_parameters' and isinstance(value, dict):
                    print(f"{prefix}🎯 {key}: [BASE_PARAMETERS DICT]")
                    for param_key, param_value in value.items():
                        print(f"{prefix}  ✅ {param_key}: {param_value}")
                elif isinstance(value, (dict, list)) and len(str(value)) > 100:
                    print(f"{prefix}📁 {key}: [COMPLEX OBJECT]")
                    if indent < 2:  # Evitar recursão muito profunda
                        print_structure(value, indent + 1, max_depth)
                else:
                    print(f"{prefix}📋 {key}: {value}")
        elif isinstance(obj, list):
            for i, item in enumerate(obj[:3]):  # Limitar a 3 itens
                print(f"{prefix}📝 [{i}]:")
                print_structure(item, indent + 1, max_depth)
        else:
            print(f"{prefix}📄 {str(obj)[:100]}{'...' if len(str(obj)) > 100 else ''}")
    
    print_structure(job_config)
    
    # Comparar com estrutura esperada
    print(f"\n📋 COMPARAÇÃO COM ESTRUTURA ESPERADA:")
    print(f"=" * 40)
    
    expected_structure = {
        "name": "dino_ingestion_...",
        "tasks": [
            {
                "task_key": "dino_ingestion_task",
                "notebook_task": {
                    "notebook_path": "/Workspace/dino/dino_ingestion_core",
                    "source": "WORKSPACE",
                    "base_parameters": {
                        "source_path": "/Volumes/.../raw",
                        "table_name": "...",
                        "catalog_name": "...",
                        "schema_name": "...",
                        "type_run": "batch"
                    }
                },
                "new_cluster": "..."
            }
        ]
    }
    
    # Verificar elementos principais
    checks = [
        ("name", job_config.get('name', '').startswith('dino_ingestion_')),
        ("tasks (array)", isinstance(job_config.get('tasks'), list) and len(job_config.get('tasks', [])) > 0),
        ("task_key", job_config.get('tasks', [{}])[0].get('task_key') == 'dino_ingestion_task'),
        ("notebook_task", 'notebook_task' in job_config.get('tasks', [{}])[0]),
        ("base_parameters", 'base_parameters' in job_config.get('tasks', [{}])[0].get('notebook_task', {}))
    ]
    
    for check_name, check_result in checks:
        status = "✅" if check_result else "❌"
        print(f"   {status} {check_name}: {check_result}")
    
    all_checks_passed = all(check[1] for check in checks)
    
    if all_checks_passed:
        print(f"\n🏆 ESTRUTURA PERFEITA!")
        print(f"✅ Todos os elementos estão presentes!")
    else:
        print(f"\n⚠️  Estrutura precisa de ajustes")
        
else:
    print("❌ Job config não disponível para análise")

## 🎯 5. Teste Final e Resumo

In [ ]:
print("🎯 RESUMO FINAL - DINO SDK v2.3.1")
print("=" * 38)

# Status geral do teste
test_results = {
    "Instalação SDK": True,
    "Criação de Job": False,
    "Base Parameters": False,
    "Estrutura Correta": False
}

if 'result' in locals():
    test_results["Criação de Job"] = result.get('success', False)
    
    if 'job_config' in result:
        job_config = result['job_config']
        tasks = job_config.get('tasks', [])
        
        if len(tasks) > 0:
            notebook_task = tasks[0].get('notebook_task', {})
            base_params = notebook_task.get('base_parameters', {})
            
            # Verificar base_parameters
            required_params = ['source_path', 'table_name', 'catalog_name', 'schema_name', 'type_run']
            has_all_params = all(param in base_params for param in required_params)
            test_results["Base Parameters"] = has_all_params
            
            # Verificar estrutura
            has_correct_structure = (
                job_config.get('name', '').startswith('dino_ingestion_') and
                len(tasks) > 0 and
                tasks[0].get('task_key') == 'dino_ingestion_task' and
                'notebook_task' in tasks[0]
            )
            test_results["Estrutura Correta"] = has_correct_structure

# Relatório final
print("📊 Resultados dos Testes:")
for test_name, passed in test_results.items():
    status = "✅ PASSOU" if passed else "❌ FALHOU"
    print(f"   {status} {test_name}")

overall_success = all(test_results.values())

if overall_success:
    print(f"\n🎉 TESTE COMPLETO APROVADO!")
    print(f"✅ DINO SDK v2.3.1 funcionando 100%!")
    print(f"🎯 Base parameters incluídos corretamente!")
    
    print(f"\n📋 Resumo da Correção:")
    print(f"   🔧 Mudou de Task/NotebookTask classes para dicionários")
    print(f"   📦 Base parameters agora incluídos via _build_job_settings()")
    print(f"   🎯 Resultado inclui job_config para debug")
    print(f"   ✅ Estrutura final compatível com referência")
    
else:
    print(f"\n⚠️  TESTE PARCIALMENTE APROVADO")
    failed_tests = [name for name, passed in test_results.items() if not passed]
    print(f"❌ Testes que falharam: {', '.join(failed_tests)}")
    
    print(f"\n💡 Próximos passos:")
    if not test_results["Base Parameters"]:
        print(f"   🔧 Verificar implementação de _build_notebook_parameters()")
    if not test_results["Estrutura Correta"]:
        print(f"   🔧 Verificar implementação de _build_job_settings()")

print(f"\n🦕 Teste DINO SDK v2.3.1 concluído!")
print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📝 Conclusão

### ✅ Correções Aplicadas na v2.3.1:

1. **Problema Identificado**: Base parameters não apareciam porque o código usava classes `Task` e `NotebookTask` diretamente
2. **Solução Implementada**: Mudou para usar `_build_job_settings()` que retorna dicionários
3. **Resultado Adicionado**: `job_config` incluído no resultado para debug
4. **Validação**: Notebook de teste para verificar presença dos base_parameters

### 🎯 Base Parameters Esperados:
```json
{
  "source_path": "/Volumes/{catalog}/{schema}/raw",
  "table_name": "{table_name}",
  "catalog_name": "{catalog_name}",  
  "schema_name": "{schema_name}",
  "type_run": "batch"
}
```

### 🚀 Como Usar:
```python
from dino_sdk import create_dino_job

result = create_dino_job(
    catalog_name="data_master_dev",
    schema_name="bronze",
    table_name="vendas_2024", 
    is_automated=True
)

# Verificar base_parameters no resultado
job_config = result['job_config']
params = job_config['tasks'][0]['notebook_task']['base_parameters']
```